# Atividade: Extração de Múltiplas Fontes e Data Wrangling

**Nome(s):**  
**Data:** 

---

### Roteiro

| Parte | Tópico |
|-------|--------|
| Parte 1 | Configuração e extração da API REST (IBGE) |
| Parte 2 | Extração de arquivo CSV (GitHub) e profiling inicial |
| Parte 3 | Diagnóstico de qualidade e aplicação de Data Wrangling |
| Parte 4 | Integração das fontes e validação final |
| Parte 5 | Análises exploratórias e reflexão |


#### Regra:

> **Escreva todo o código por conta própria**. Consulte a documentação das bibliotecas sempre que necessário.

---
## Contexto de Negócio

Você trabalha como **Engenheiro(a) de Dados** em uma startup de inteligência de
mercado imobiliário. A empresa quer lançar uma plataforma que ajude investidores
a comparar o potencial de diferentes municípios brasileiros.

Para isso, você precisa construir uma **base de dados integrada de municípios**,
combinando informações geográficas, administrativas e demográficas provenientes
de **duas fontes distintas**:

| Fonte | Tipo | Dados disponíveis |
|-------|------|-------------------|
| **API IBGE Localidades** | REST API (JSON) | Hierarquia geográfica, mesorregião, microrregião, UF, região |
| **CSV de Municípios Brasileiros** | Arquivo CSV (GitHub) | Coordenadas geográficas, indicador de capital, nomes e códigos de subdivisões |

Ao combinar as duas fontes, você descobrirá **inconsistências reais** entre elas e deverá aplicar técnicas de **Data Wrangling** para criar uma base integrada
e confiável.


---
## Fontes de Dados

### Fonte 1 - API IBGE Localidades (REST/JSON)

O IBGE disponibiliza uma API pública e gratuita sem necessidade de cadastro ou autenticação.
Documentação completa: **https://servicodados.ibge.gov.br/api/docs/localidades**

Endpoints que você utilizará:

| Endpoint | URL | Retorna |
|----------|-----|---------|
| Municípios | `https://servicodados.ibge.gov.br/api/v1/localidades/municipios` | Lista de todos os 5.570 municípios |
| Estados | `https://servicodados.ibge.gov.br/api/v1/localidades/estados` | 27 UFs com região |
| Regiões | `https://servicodados.ibge.gov.br/api/v1/localidades/regioes` | 5 regiões do Brasil |

A resposta do endpoint de municípios é um **array JSON** com objetos aninhados no formato:
```json
{
  "id": 5002704,
  "nome": "Água Clara",
  "microrregiao": {
    "id": 50005,
    "nome": "Três Lagoas",
    "mesorregiao": {
      "id": 5001,
      "nome": "Leste de Mato Grosso do Sul",
      "UF": {
        "id": 50,
        "sigla": "MS",
        "nome": "Mato Grosso do Sul",
        "regiao": { "id": 5, "sigla": "CO", "nome": "Centro-Oeste" }
      }
    }
  }
}
```
> **Atenção:** A resposta é um JSON **aninhado** (nested). Você precisará *achatar* (flatten)
> essa estrutura para carregá-la em um DataFrame tabular.

---

### Fonte 2 - CSV de Municípios Brasileiros (GitHub)

URL direta para download:
```
https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv
```

Colunas disponíveis no arquivo:

| Coluna | Descrição |
|--------|----------|
| `codigo_ibge` | Código IBGE do município (7 dígitos) |
| `nome` | Nome do município |
| `latitude` | Latitude do centroide |
| `longitude` | Longitude do centroide |
| `capital` | 1 se é capital estadual, 0 caso contrário |
| `codigo_uf` | Código numérico da UF |
| `codigo_mesorregiao` | Código da mesorregião |
| `nome_mesorregiao` | Nome da mesorregião |
| `codigo_microrregiao` | Código da microrregião |
| `nome_microrregiao` | Nome da microrregião |


---
# Parte 1 - Configuração e Extração via API REST

### Tarefa 1.1 - Instalação e importações

Instale e importe as bibliotecas necessárias para a atividade.  
Você precisará de: `requests` (requisições HTTP), `pandas`, `numpy`, `json`.


In [ ]:
# Tarefa 1.1 - Instale e importe as bibliotecas necessárias

import requests
import pandas as pd
import numpy as np
import json

### Tarefa 1.2 - Extrair dados dos municípios via API

Faça uma requisição GET ao endpoint de municípios do IBGE e armazene a resposta.  
Verifique o status code da resposta antes de prosseguir.

> **Dica:** Use `requests.get(url)` e verifique se `response.status_code == 200`.  
> O conteúdo pode ser convertido para lista Python com `response.json()`.


In [ ]:
# Tarefa 1.2 - Faça a requisição à API e inspecione a resposta

url_minucipios = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
response = requests.get(url_minucipios)
print(response.status_code)  # Verificar se a requisição foi bem-sucedida
print(response.headers)  # Inspecionar os cabeçalhos da resposta

### Tarefa 1.3 - Achatar o JSON aninhado (Flatten)

A resposta da API é um JSON aninhado com vários níveis. Você precisa transformá-lo
em uma estrutura tabular (um registro por município) com as seguintes colunas:

| Coluna desejada | Caminho no JSON |
|-----------------|----------------|
| `id_municipio` | `id` |
| `nome_municipio` | `nome` |
| `id_microrregiao` | `microrregiao.id` |
| `nome_microrregiao` | `microrregiao.nome` |
| `id_mesorregiao` | `microrregiao.mesorregiao.id` |
| `nome_mesorregiao` | `microrregiao.mesorregiao.nome` |
| `sigla_uf` | `microrregiao.mesorregiao.UF.sigla` |
| `nome_uf` | `microrregiao.mesorregiao.UF.nome` |
| `id_uf` | `microrregiao.mesorregiao.UF.id` |
| `sigla_regiao` | `microrregiao.mesorregiao.UF.regiao.sigla` |
| `nome_regiao` | `microrregiao.mesorregiao.UF.regiao.nome` |


> **Dica 1:** Itere pela lista de municípios e extraia cada campo do dicionário aninhado.  
> **Dica 2:** A função `pd.json_normalize()` pode simplificar esse processo para objetos simples.


In [ ]:
# Tarefa 1.3 - Achate o JSON e crie um DataFrame com as colunas solicitadas
data = response.json()  # Converter a resposta JSON em um objeto Python
# Criar um DataFrame a partir dos dados JSON
df = pd.json_normalize(data)
print(df.head())  # Exibir as primeiras linhas do DataFrame para verificar a estrutura


### Tarefa 1.4 - Verificação inicial da extração

Após criar o DataFrame, responda:
- Quantos municípios foram extraídos? (o Brasil possui 5.570)
- Quais são os tipos de cada coluna?
- Existem valores nulos?
- Exiba as primeiras 5 linhas do DataFrame.


In [ ]:
# Tarefa 1.4 - Verifique a integridade do DataFrame extraído da API
count_municipios = df['nome'].shape[0] - 1 # Contar o número de linhas (municipios)
print(f"Número de municípios: {count_municipios}")

print(df.info())  # Verificar o tipo de dados e a presença de valores nulos

# Verificar se há valores nulos em colunas importantes
print(df.isnull().sum())

df.describe() # Obter estatísticas descritivas para colunas numéricas

---
# Parte 2 - Extração do CSV e Data Profiling

### Tarefa 2.1 - Extrair o CSV diretamente da URL

Carregue o arquivo CSV de municípios brasileiros diretamente da URL do GitHub
em um DataFrame.

> **Dica:** O `pandas.read_csv()` aceita URLs diretamente como argumento.


In [ ]:
# Tarefa 2.1 - Carregue o CSV da URL diretamente para um DataFrame
url_csv = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"

df_csv = pd.read_csv(url_csv)
df_csv.head()  # Exibir as primeiras linhas do DataFrame para verificar o carregamento

### Tarefa 2.2 - Data Profiling do CSV

Realize um profiling completo do DataFrame do CSV. Para cada coluna, levante:

- Tipo de dado atual
- Quantidade e percentual de valores nulos
- Quantidade de valores únicos (cardinalidade)
- Para colunas numéricas: mínimo, máximo, média e desvio padrão
- Para colunas categóricas/texto: os 5 valores mais frequentes
- Existência de valores duplicados por código IBGE


In [ ]:
# Tarefa 2.2a - Análise de nulos e cardinalidade de cada coluna
df_csv.isnull().sum()




In [ ]:
# Tarefa 2.2a - Análise de nulos e cardinalidade de cada coluna
df_csv.nunique()

In [ ]:
# Tarefa 2.2b - Estatísticas descritivas das colunas numéricas
df_csv.describe().T



In [ ]:
# Tarefa 2.2c - Distribuição de frequência das colunas categóricas (top 5 valores)
df_csv['fuso_horario'].value_counts().head(5)

In [ ]:
# Tarefa 2.2c - Distribuição de frequência das colunas categóricas (top 5 valores)
df_csv['nome'].value_counts().head(5)


In [ ]:
# Tarefa 2.2d - Verifique duplicatas pelo código IBGE


### Tarefa 2.3 - Data Profiling do DataFrame da API

Realize o mesmo profiling da Tarefa 2.2, agora para o DataFrame extraído da API IBGE.
Preste atenção especial às colunas de texto: elas virão com acentuação?
Os nomes de cidades estão padronizados?

- Tipo de dado atual
- Quantidade e percentual de valores nulos
- Quantidade de valores únicos (cardinalidade)
- Para colunas numéricas: mínimo, máximo, média e desvio padrão
- Para colunas categóricas/texto: os 5 valores mais frequentes
- Existência de valores duplicados por código IBGE


In [ ]:
# Tarefa 2.2a - Análise de nulos e cardinalidade de cada coluna
df_csv.info()

df_csv.isnull()

In [ ]:
# Tarefa 2.2b - Estatísticas descritivas das colunas numéricas
df_csv.describe()

In [ ]:
# Tarefa 2.2c - Distribuição de frequência das colunas categóricas (top 5 valores)
df_csv['fuso_horario'].value_counts().head(5)

In [ ]:
# Tarefa 2.2c - Distribuição de frequência das colunas categóricas (top 5 valores)
df_csv['nome'].value_counts().head(5)


In [ ]:
# Tarefa 2.2d - Verifique duplicatas pelo código IBGE
print(df_csv['codigo_ibge'].duplicated().sum() )

# Tarefa 2.3c - Distribuição de frequência das colunas categóricas (top 5 valores)

def top_frequent_values():
    for column in df.select_dtypes(include=['object']).columns:
        print(f"Coluna: {column}")
        print(df_csv[column].value_counts().head(5))  # Exibir os 5 valores mais frequentes
        print("\n")
        
top_frequent_values()

In [ ]:
# Tarefa 2.3a - Análise de nulos e cardinalidade de cada coluna
df.info()

# Verifica a quantidade de valores nulos em cada coluna
df.isnull().sum()

In [ ]:
# Tarefa 2.3b - Estatísticas descritivas das colunas numéricas

df.describe().T

In [ ]:
# Tarefa 2.3c - Distribuição de frequência das colunas categóricas (top 5 valores)

def top_frequent_values():
    for column in df.select_dtypes(include=['object']).columns:
        print(f"Coluna: {column}")
        print(df_csv[column].value_counts().head(5))  # Exibir os 5 valores mais frequentes
        print("\n")
        
top_frequent_values()

In [ ]:
# Tarefa 2.3d - Verifique duplicatas pelo código IBGE
print(df.duplicated(subset=['id'], keep=False).sum()) 


---
# Parte 3 - Diagnóstico de Qualidade e Data Wrangling

### Tarefa 3.1 - Diagnóstico comparativo entre as duas fontes

Compare os dois DataFrames para identificar possíveis inconsistências **entre** eles.  
Investigue:

1. Os **dois datasets possuem o mesmo número de municípios?** Se não, quais estão faltando em um ou outro?
2. Os **nomes dos municípios** são idênticos nas duas fontes, ou existem variações de grafia?
3. Os **nomes de mesorregiões e microrregiões** são consistentes entre as duas fontes?
4. O campo `codigo_ibge` do CSV corresponde ao campo `id_municipio` da API?
   Atenção: o CSV usa 7 dígitos e a API pode usar formato diferente.

> **Dica:** Faça um merge entre os dois DataFrames por código IBGE e analise os registros
> que não encontraram correspondência (`how='outer'` + verificar NaNs).


In [ ]:
# Normalização de colunas para garantir consistência entre os DataFrames
import unicodedata

df_csv = df_csv.rename(columns={"codigo_ibge": "id"})

def normalizar(texto):
    if pd.isna(texto):
        return texto
        
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    
    return texto

df_csv["nome"] = df_csv["nome"].apply(normalizar)
df["nome"] = df["nome"].apply(normalizar)

df_merge = df_csv.merge(df, on="id", suffixes=("_csv", "_municipios"))
diferentes = df_merge[df_merge["nome_csv"] != df_merge["nome_municipios"]]

In [ ]:
# Tarefa 3.1a - Compare o volume de municípios nas duas fontes
print(f"Volume de municípios no DataFrame do CSV: {df_csv.shape[0]}")
print(f"Volume de municípios no DataFrame da API: {df.shape[0]}")

In [ ]:
# Tarefa 3.1b - Identifique municípios presentes em uma fonte mas ausentes na outra
print(diferentes[["id", "nome_csv", "nome_municipios"]])

In [ ]:
# Tarefa 3.1c - Compare os nomes de mesorregiões e microrregiões entre as fontes
df_merge_meso = df_csv.merge(df, on="id", suffixes=("_csv", "_municipios"))
diferentes_meso = df_merge_meso[df_merge_meso["mesorregiaoo_csv"] != df_merge_meso["mesorregiao_municipios"]]
print(diferentes_meso[["id", "mesorregiaoo_csv", "mesorregiao_municipios"]])


### Tarefa 3.2 - Relatório de problemas de qualidade encontrados

Com base nas análises anteriores, preencha a tabela abaixo:

| # | Fonte | Campo(s) | Dimensão de Qualidade | Problema Encontrado | Frequência | Ação Planejada |
|---|-------|----------|-----------------------|---------------------|------------|----------------|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |
| 4 | | | | | | |
| 5 | | | | | | |

**Dimensões de qualidade:** Completude, Unicidade, Validade, Consistência, Precisão, Atualidade


### Tarefa 3.3 - Aplicação das Técnicas de Data Wrangling

Para cada problema identificado, aplique a técnica de tratamento mais adequada.  
Use células separadas para cada tipo de tratamento e **adicione um comentário**
explicando a decisão tomada.

**Lembre-se:** Trabalhe em cópias dos DataFrames originais.


In [ ]:
# Tarefa 3.3a - Corrija os tipos de dados incorretos em ambos os DataFrames


In [ ]:
# Tarefa 3.3b - Padronize strings (encoding, case, espaços extras, caracteres especiais)


In [ ]:
# Tarefa 3.3c - Trate os valores nulos encontrados


In [ ]:
# Tarefa 3.3d - Valide as coordenadas geográficas do CSV
# Latitude válida: entre -90 e 90
# Longitude válida: entre -180 e 180
# Para o Brasil: lat entre -34 e 6; lon entre -74 e -28


In [ ]:
# Tarefa 3.3e - Aplique demais correções identificadas no diagnóstico


---
# Parte 4 - Integração das Fontes e Validação Final

### Tarefa 4.1 - Preparar chaves de integração

Para unir as duas fontes, você precisa de uma **chave de integração comum**.  
O CSV usa o código IBGE com 7 dígitos. A API retorna o campo `id_municipio`.

Verifique se os dois campos têm o mesmo formato e, se necessário, ajuste
um deles para garantir que o join funcionará corretamente.

> **Atenção:** Verifique o número de dígitos. O código IBGE completo tem 7 dígitos.
> Se a API retornar 6, pode ser necessário adicionar o dígito verificador ou vice-versa.


In [ ]:
# Tarefa 4.1 - Verifique e ajuste as chaves de integração


### Tarefa 4.2 - Integrar as duas fontes

Realize o merge entre os dois DataFrames tratados usando a chave de integração.  

O DataFrame integrado final deve conter, para cada município:

| Coluna | Origem |
|--------|--------|
| `codigo_ibge` | Chave de integração |
| `nome_municipio` | API (ou CSV — escolha e justifique) |
| `latitude` | CSV |
| `longitude` | CSV |
| `capital` | CSV |
| `sigla_uf` | API |
| `nome_uf` | API |
| `sigla_regiao` | API |
| `nome_regiao` | API |
| `nome_mesorregiao` | API (ou CSV — justifique a escolha) |
| `nome_microrregiao` | API (ou CSV — justifique a escolha) |

> **Dica:** Use `pd.merge()` com `how='inner'` para início.  
> Depois verifique quantos registros foram perdidos e investigue os casos.


In [ ]:
# Tarefa 4.2 - Realize o merge das duas fontes e construa o DataFrame integrado


### Tarefa 4.3 - Validação final do dataset integrado

Antes de declarar o dataset como pronto para carga, valide:

1. O número de municípios está correto (esperado: 5.570)?
2. Nenhum campo obrigatório possui nulos?
3. Os valores de `sigla_uf` cobrem todos os 27 estados?
4. Os valores de `sigla_regiao` cobrem exatamente 5 regiões (N, NE, CO, SE, S)?
5. As coordenadas de todos os municípios estão dentro dos limites geográficos do Brasil?
6. O campo `capital` possui exatamente 27 municípios marcados como capital (1 por UF)?

Escreva **assertions** (verificações programáticas) para cada item acima.


In [ ]:
# Tarefa 4.3 - Escreva verificações de qualidade para o dataset integrado


---
# Parte 5 - Análises Exploratórias e Reflexão

### Tarefa 5.1 - Análises com o dataset integrado

Responda às perguntas abaixo usando o dataset integrado e tratado:


**Pergunta A:** Quantos municípios existem em cada região do Brasil?
Qual região concentra o maior número de municípios?


In [ ]:
# Pergunta A


**Pergunta B:** Qual estado possui o maior número de mesorregiões?
E o maior número de microrregiões?


In [ ]:
# Pergunta B


**Pergunta C:** Identifique o município mais ao norte, mais ao sul, mais a leste
e mais a oeste do Brasil usando as coordenadas geográficas.


In [ ]:
# Pergunta C


**Pergunta D:** Calcule o centroide geográfico (latitude e longitude médias)
de cada região do Brasil. Qual região tem o centroide mais próximo do equador?


In [ ]:
# Pergunta D


---
### Tarefa 5.2 - Reflexão sobre o processo

Responda às perguntas abaixo com base na sua experiência ao longo desta atividade:

**1. Qual das duas fontes apresentou maior número de problemas de qualidade?
Que fatores podem explicar essa diferença?**

*Sua resposta:*

---

**2. Qual técnica de Data Wrangling você considerou mais desafiadora de implementar e por quê?**

*Sua resposta:*

---

**3. Ao fazer o merge entre as duas fontes, alguns municípios podem não ter encontrado
correspondência. O que você faria com esses registros em um projeto real de DW?
Descartaria ou investigaria? Como documentaria essa decisão?**

*Sua resposta:*

---

**4. Se você precisasse atualizar este dataset mensalmente (novos municípios criados,
renomeações, mudanças de divisão administrativa), qual estratégia de extração
(full load, incremental ou CDC) seria mais adequada para cada fonte? Justifique.**

*Sua resposta:*

---

**5. Este dataset de municípios poderia ser usado como uma dimensão em um DW?
Se sim, qual seria o nome dessa dimensão, quais colunas ela incluiria,
e quais tabelas fato poderiam se relacionar com ela?**

*Sua resposta:*
